# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset (mass spectrometry of extracellular vesicles) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The dataset may contain multiple record sets, each with its own fields (analogous to table columns). We will list the available record sets and examine their structure.

**Note:** All references to record sets, fields, and columns will use their `@id` values as recommended.

In [ ]:
# List available record sets and their field IDs
record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print("  Fields (by @id):")
    for f in rs.fields:
        print(f"    - {f.id} ({f.name})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s found in the overview above.

We will extract all record sets and load them into pandas DataFrames using the `@id` references.

In [ ]:
# Extract data from each record set
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]
for record_set_id in record_set_ids:
    # records() yields dicts mapping field @id to values
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display columns for the first record set
example_rs_id = record_set_ids[0] if record_set_ids else None
if example_rs_id:
    print(f"Columns for record set '@id': {example_rs_id}\n{dataframes[example_rs_id].columns.tolist()}")
    display(dataframes[example_rs_id].head())
else:
    print('No record sets found in metadata.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

Below, we select a record set and field for demo EDA. You can adjust the `record_set_id`, `numeric_field_id`, and `group_field_id` as needed for your analysis.

In [ ]:
# Pick a record set and fields suitable for EDA. Edit these IDs as needed based on printed overview above.
record_set_id = example_rs_id
# Assume the record set has a numeric field, e.g. protein coverage or abundance. Replace with real field @id from overview.
numeric_field_id = None
group_field_id = None

# Inspect field IDs to suggest typical numeric fields
if record_set_id:
    for f in [f for f in [f for f in metadata.record_sets if f.id==record_set_id][0].fields]:
        # Pick common numeric-like field id heuristically
        if 'cover' in f.id.lower() or 'abundance' in f.id.lower() or 'count' in f.id.lower() or 'mw' in f.id.lower():
            numeric_field_id = f.id
        # For grouping, try 'sample', 'group', etc.
        if group_field_id is None and ('sample' in f.id.lower() or 'group' in f.id.lower()):
            group_field_id = f.id

# Fallbacks in case not detected
if not numeric_field_id:
    numeric_field_id = dataframes[record_set_id].columns[0]  # fallback: use first column
if not group_field_id:
    group_field_id = dataframes[record_set_id].columns[1] if len(dataframes[record_set_id].columns) > 1 else None

print(f"Using record set: {record_set_id}")
print(f"Numeric field: {numeric_field_id}")
print(f"Group field: {group_field_id}")

df = dataframes[record_set_id]
# Convert the numeric field to float if possible
if numeric_field_id in df.columns:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
filtered_df = df[df[numeric_field_id] > threshold]

print(f"Filtered records in '{record_set_id}' with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() else 1)
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean over {numeric_field_id}):")
    display(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships in the chosen record set and fields.

We'll create a histogram for the numeric field and, if applicable, a bar chart grouped by the grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field_id and numeric_field_id in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# Bar plot grouped by group_field_id (if present)
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10, 5))
    group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
    group_means.plot.bar()
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to programmatically access and analyze the FAIR^2 mass spectrometry dataset using the Croissant schema and the `mlcroissant` library. We explored the dataset structure, extracted tables using record set and field `@id`s, and performed a simple data analysis and visualization.

You can further adapt the code sections for deeper analysis or for other record sets/fields according to your research questions.